In [7]:
import os
import time
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding
from torch.utils.data import DataLoader
from chop import MaseGraph
from chop.tools import get_tokenized_dataset
import inspect



In [8]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    props = torch.cuda.get_device_properties(0)
    print("Using CUDA device:", props.name)
else:
    DEVICE = torch.device("cpu")
    print("CUDA not available — using CPU")

checkpoint = "DeepWokLab/bert-tiny"
tokenizer_checkpoint = "DeepWokLab/bert-tiny"
dataset_name = "imdb"

dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)


INFO     Tokenizing dataset imdb with AutoTokenizer for DeepWokLab/bert-tiny.


Using CUDA device: NVIDIA GeForce RTX 4080


In [3]:
eval_ds = dataset["test"].remove_columns(["text", "token_type_ids"]).rename_column("label", "labels")

eval_dataloader = DataLoader(
    eval_ds,
    batch_size=64,
    shuffle=False,
    collate_fn=DataCollatorWithPadding(tokenizer),
)

@torch.no_grad()
def eval_accuracy(model, dataloader, device="cuda"):
    model.eval()
    correct, total = 0, 0
    for batch in tqdm(dataloader, desc="Eval Accuracy"):
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        logits = out["logits"] if isinstance(out, dict) else out.logits
        correct += (logits.argmax(dim=-1) == batch["labels"]).sum().item()
        total += batch["labels"].size(0)
    acc = correct / total
    print(f"Accuracy: {acc * 100:.2f}% ({correct}/{total})")
    return acc

@torch.no_grad()
def eval_speed(model, dataloader, device="cuda", num_batches=100, warmup=10):
    model.eval()
    batches = list(dataloader)[:warmup + num_batches]
    # warmup
    for b in batches[:warmup]:
        model(**{k: v.to(device) for k, v in b.items()})
    if device.type == "cuda":
        torch.cuda.synchronize()

    # timed
    times, samples = [], 0
    for b in batches[warmup:]:
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(**{k: v.to(device) for k, v in b.items()})
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
        samples += b["input_ids"].size(0)
    avg_per_batch_ms = np.mean(times) * 1000
    avg_per_sample_ms = (sum(times) / samples) * 1000
    total = sum(times)
    print(f"{samples} samples in {total:.4f}s")
    print(f"Throughput:      {samples / total:.1f} samples/sec")
    print(f"Avg batch:       {np.mean(times) * 1000:.2f} ms")
    print(f"Avg per-sample:  {total / samples * 1000:.2f} ms")
    return [avg_per_batch_ms, avg_per_sample_ms]



In [19]:

class EarlyExitBerttiny(nn.Module):
    def __init__(self, original_model, threshold=0.9):
        super().__init__()
        # Extracted submodules 
        self.bert = original_model.bert
        self.classifier = original_model.classifier
        self.threshold = threshold
        
        # In a generic module graph, children might not be inside a ModuleList with length
        self.encoder_layers = nn.ModuleList(list(self.bert.encoder.layer.children()))
        self.num_layers = len(self.encoder_layers)

    def forward(self, input_ids, attention_mask, labels=None):
        device = input_ids.device
        batch_size = input_ids.size(0)

        # Standard representation mask
        extended_attention_mask = self.bert.get_extended_attention_mask(attention_mask, input_ids.size(), device)
        
        # Step 1: Embeddings
        hidden_states = self.bert.embeddings(input_ids=input_ids)
        
        final_logits = torch.zeros(batch_size, self.classifier.out_features, device=device)
        
        # We index elements to dynamically shrink batch sizes across layers
        active_mask = extended_attention_mask
        active_indices = torch.arange(batch_size, device=device)

        for i, layer_module in enumerate(self.encoder_layers):
            print(f"Layer {i+1}/{self.num_layers} - Active samples: {active_indices.size(0)} Layer: {layer_module.__class__.__name__}")
            
            # Step 2: Layer Forward
            layer_outputs = layer_module(hidden_states, attention_mask=active_mask)
            hidden_states = layer_outputs[0]
            
            # Step 3: Compute confidence through pooling and classification
            cls_token = hidden_states[:, 0]
            pooled_output = self.bert.pooler.dense(cls_token)
            pooled_output = self.bert.pooler.activation(pooled_output)
            logits = self.classifier(pooled_output)
            
            if i == self.num_layers - 1:
                final_logits[active_indices] = logits
                break

            probs = F.softmax(logits, dim=-1)
            max_probs, _ = torch.max(probs, dim=-1)
            
            # Step 4: Early Exit Criteria
            confident = max_probs >= self.threshold
            
            # Store completed outputs
            confident_indices = active_indices[confident]
            final_logits[confident_indices] = logits[confident]
            
            not_confident = ~confident
            if not not_confident.any():
                break
            
            # Dynamically shrink tensor blocks to only compute unconfident targets in next layer
            active_indices = active_indices[not_confident]
            hidden_states = hidden_states[not_confident]
            active_mask = active_mask[not_confident]
        

        return {"logits": final_logits}
    


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EarlyExitMixin:
    """
    A mixin that provides universal early exit tracking, confidence 
    evaluation, and classifier routing.
    """
    def init_early_exit(self, original_model, threshold):
        self.threshold = threshold
        self.num_labels = original_model.config.num_labels
        self.pooler = None # Will be overridden by subclass if applicable
        self.classifier = None # Will be overridden by subclass

    def compute_logits(self, hidden_states):
        """Universal routing for classifiers (Handles [CLS] vs Pooler)"""
        if self.pooler is not None:
            return self.classifier(self.pooler(hidden_states))
        
        # Fallback to standard [CLS] slicing if no pooler exists
        try:
            return self.classifier(hidden_states[:, 0])
        except Exception:
            return self.classifier(hidden_states) # 3D fallback

    # def compute_logits(self, hidden_states):
    #     # RoBERTa's classifier (RobertaClassificationHead) 
    #     # internally does: x = features[:, 0, :]
    #     # So we MUST pass the 3D hidden_states [Batch, Seq, Hidden]
    #     return self.classifier(hidden_states)


    def evaluate_confidence(self, logits, active_indices, final_logits):
        """Checks threshold, saves confident predictions, returns the leftover mask."""
        probs = F.softmax(logits, dim=-1)
        max_p, _ = torch.max(probs, dim=-1)
        confident = max_p >= self.threshold
        
        # Lock in the confident predictions
        final_logits[active_indices[confident]] = logits[confident]
        
        # Return boolean mask of who is NOT confident (who moves to the next layer)
        return ~confident

In [61]:
import torch.nn as nn

class RobertaEarlyExit(nn.Module, EarlyExitMixin):
    def __init__(self, original_model, threshold=0.75):
        super().__init__()
        self.init_early_exit(original_model, threshold)
        
        # We assume standard HuggingFace RoBERTa model structure now
        self.embeddings = original_model.roberta.embeddings
        self.layers = nn.ModuleList(original_model.roberta.encoder.layer)
        self.classifier = original_model.classifier
        
        self.num_layers = len(self.layers)

    def compute_logits(self, hidden_states):
        # RoBERTa classifier internally slicing [:, 0, :]
        return self.classifier(hidden_states)

    def forward(self, input_ids, attention_mask, **kwargs):
        device = input_ids.device
        batch_size = input_ids.size(0)

        # 1. Setup
        hidden_states = self.embeddings(input_ids=input_ids)
        
        # Create the standard 4D mask RoBERTa needs
        extended_attention_mask = attention_mask[:, None, None, :]
        extended_attention_mask = extended_attention_mask.to(dtype=hidden_states.dtype)
        active_mask = (1.0 - extended_attention_mask) * -10000.0

        final_logits = torch.zeros(batch_size, self.num_labels, device=device)
        active_indices = torch.arange(batch_size, device=device)

        # 2. Execution Loop
        for i, layer in enumerate(self.layers):
            layer_outputs = layer(hidden_states, attention_mask=active_mask)
            hidden_states = layer_outputs[0]

            logits = self.compute_logits(hidden_states)

            if i == self.num_layers - 1:
                final_logits[active_indices] = logits
                break

            not_confident = self.evaluate_confidence(logits, active_indices, final_logits)
            if not not_confident.any(): 
                break 
                
            # 3. Shrink
            active_indices = active_indices[not_confident]
            hidden_states = hidden_states[not_confident]
            active_mask = active_mask[not_confident]

        return {"logits": final_logits}

In [ ]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# class DebertaEarlyExit(nn.Module):
#     def __init__(self, original_model, threshold=0.75):
#         super().__init__()
#         self.threshold = threshold
#         self.num_labels = original_model.config.num_labels
        
#         # Pulling out the core components
#         self.deberta = original_model.deberta
#         self.embeddings = self.deberta.embeddings
#         self.layers = self.deberta.encoder.layer
        
#         # The classifier head (contains dropout and the linear layer)
#         self.classifier = original_model.classifier
        
#         # DeBERTa-v3 specific: The encoder has an internal method to get relative pos
#         self.encoder = self.deberta.encoder
#         self.num_layers = len(self.layers)

#     def compute_logits(self, hidden_states):
#         """
#         In DeBERTa, the classifier head is designed to take the [CLS] token.
#         We slice [:, 0, :] to get the representation of the first token.
#         """
#         cls_state = hidden_states[:, 0, :]
#         return self.classifier(cls_state)

#     def evaluate_confidence(self, logits, active_indices, final_logits):
#         """
#         Determines which samples in the current batch are ready to exit.
#         """
#         probs = F.softmax(logits, dim=-1)
#         max_p, _ = torch.max(probs, dim=-1)
#         confident = max_p >= self.threshold
        
#         # Save results for confident samples
#         if confident.any():
#             final_logits[active_indices[confident]] = logits[confident]
        
#         # Return mask of samples that are NOT confident and need to continue
#         return ~confident

#     def forward(self, input_ids, attention_mask=None, **kwargs):
#         device = input_ids.device
#         batch_size = input_ids.size(0)

#         # 1. Prepare Inputs
#         # DeBERTa's embeddings handle position_ids internally if not provided
#         hidden_states = self.embeddings(input_ids=input_ids)
        
#         # Generate the specific attention mask DeBERTa expects (usually 4D)
#         active_mask = self.encoder.get_attention_mask(attention_mask)
        
#         # Calculate Relative Positions (broadcastable [1, Seq, Seq])
#         relative_pos = self.encoder.get_rel_pos(hidden_states)
#         rel_embeddings = self.encoder.rel_embeddings.weight

#         # Track outputs
#         final_logits = torch.zeros(batch_size, self.num_labels, device=device)
#         active_indices = torch.arange(batch_size, device=device)

#         # 2. Layer-by-Layer Execution
#         for i, layer in enumerate(self.layers):
#             # Process through the current transformer layer
#             layer_outputs = layer(
#                 hidden_states,
#                 attention_mask=active_mask,
#                 relative_pos=relative_pos,
#                 rel_embeddings=rel_embeddings
#             )
#             hidden_states = layer_outputs[0]

#             # Generate logits for the current state
#             logits = self.compute_logits(hidden_states)

#             # If it's the final layer, we MUST exit
#             if i == self.num_layers - 1:
#                 final_logits[active_indices] = logits
#                 break

#             # Check for early exit
#             not_confident = self.evaluate_confidence(logits, active_indices, final_logits)
            
#             # If everyone is confident, we stop early entirely
#             if not not_confident.any():
#                 break
            
#             # 3. Shrink Tensors for the next layer
#             # We only shrink batch-dimension tensors. relative_pos [1, S, S] stays.
#             active_indices = active_indices[not_confident]
#             hidden_states = hidden_states[not_confident]
#             active_mask = active_mask[not_confident]

#         return {"logits": final_logits}

In [5]:
class BertEarlyExit(nn.Module, EarlyExitMixin):
    def __init__(self, original_model, threshold=0.75):
        super().__init__()
        # 1. Initialize the shared mixin logic
        self.init_early_exit(original_model, threshold)
        
        # 2. Hardcode the paths for BERT
        self.embeddings = original_model.bert.embeddings
        self.layers = original_model.bert.encoder.layer
        self.pooler = original_model.bert.pooler
        self.classifier = original_model.classifier
        
        self.base_model = original_model.bert
        self.num_layers = len(self.layers)

    def forward(self, input_ids, attention_mask, **kwargs):
        device = input_ids.device
        batch_size = input_ids.size(0)

        # 1. Setup
        hidden_states = self.embeddings(input_ids=input_ids)
        # Create the standard 4D mask BERT needs
        active_mask = self.base_model.get_extended_attention_mask(attention_mask, input_ids.size(), device)
        
        final_logits = torch.zeros(batch_size, self.num_labels, device=device)
        active_indices = torch.arange(batch_size, device=device)

        # 2. Execution Loop
        for i, layer in enumerate(self.layers):
            layer_outputs = layer(hidden_states, attention_mask=active_mask)
            hidden_states = layer_outputs[0]

            # The mixin automatically routes through self.pooler then self.classifier
            logits = self.compute_logits(hidden_states)

            if i == self.num_layers - 1:
                final_logits[active_indices] = logits
                break

            # Check threshold and get mask of who failed to exit
            not_confident = self.evaluate_confidence(logits, active_indices, final_logits)
            
            if not not_confident.any(): 
                break # Everyone exited!
                
            # 3. Shrink specific to standard BERT
            active_indices = active_indices[not_confident]
            hidden_states = hidden_states[not_confident]
            active_mask = active_mask[not_confident]

        return {"logits": final_logits}

In [ ]:
# class ModernBertEarlyExit(nn.Module):
#     def __init__(self, original_model, threshold=0.8):
#         super().__init__()
#         self.threshold = threshold
#         self.num_labels = original_model.config.num_labels
#         self.config = original_model.config
        
#         # Base components
#         self.base_model = original_model.model 
#         self.embeddings = self.base_model.embeddings
#         self.layers = self.base_model.layers
#         self.final_norm = self.base_model.final_norm
        
#         self.head = original_model.head
#         self.drop = original_model.drop # Dropout layer
#         self.classifier = original_model.classifier
        
#         self.num_layers = len(self.layers)

#     def compute_logits(self, hidden_states, attention_mask):
#         # 1. Apply final_norm
#         hidden_states = self.final_norm(hidden_states)
        
#         # 2. Pool
#         if self.config.classifier_pooling == "cls":
#             pooled = hidden_states[:, 0]
#         elif self.config.classifier_pooling == "mean":
#             # attention_mask is 1D or 2D here? It's the original 2D mask.
#             mask_expanded = attention_mask.unsqueeze(-1).to(hidden_states.dtype)
#             pooled = (hidden_states * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1e-9)
#         else:
#             pooled = hidden_states[:, 0]

#         # 3. Head and Classifier
#         pooled = self.head(pooled)
#         if hasattr(self, 'drop') and self.drop is not None:
#             pooled = self.drop(pooled)
#         return self.classifier(pooled)

#     def evaluate_confidence(self, logits, active_indices, final_logits):
#         probs = F.softmax(logits, dim=-1)
#         max_p, _ = torch.max(probs, dim=-1)
#         confident = max_p >= self.threshold
#         if confident.any():
#             final_logits[active_indices[confident]] = logits[confident]
#         return ~confident

#     def forward(self, input_ids, attention_mask, **kwargs):
#         device = input_ids.device
#         batch_size, seq_len = input_ids.size()

#         # 1. Setup Embeddings
#         hidden_states = self.embeddings(input_ids=input_ids)
        
#         # 2. Setup Attention Mask
#         # Use ModernBert's internal mask logic
#         global_mask, sliding_window_mask = self.base_model._update_attention_mask(
#             attention_mask, output_attentions=False
#         )
        
#         active_raw_mask = attention_mask
#         active_global_mask = global_mask
#         active_sliding_window_mask = sliding_window_mask
        
#         # --- Create Position IDs ---
#         position_ids = torch.arange(seq_len, dtype=torch.long, device=device)
#         position_ids = position_ids.unsqueeze(0).expand(batch_size, -1)
#         active_pos_ids = position_ids

#         final_logits = torch.zeros(batch_size, self.num_labels, device=device)
#         active_indices = torch.arange(batch_size, device=device)

#         # 3. Execution Loop
#         for i, layer in enumerate(self.layers):
#             layer_outputs = layer(
#                 hidden_states, 
#                 attention_mask=active_global_mask,
#                 sliding_window_mask=active_sliding_window_mask,
#                 position_ids=active_pos_ids 
#             )
#             hidden_states = layer_outputs[0]

#             logits = self.compute_logits(hidden_states, active_raw_mask)

#             if i == self.num_layers - 1:
#                 final_logits[active_indices] = logits
#                 break

#             not_confident = self.evaluate_confidence(logits, active_indices, final_logits)
#             if not not_confident.any(): 
#                 break
                
#             # 4. Shrink all batch-dependent tensors
#             active_indices = active_indices[not_confident]
#             hidden_states = hidden_states[not_confident]
#             active_raw_mask = active_raw_mask[not_confident]
#             active_global_mask = active_global_mask[not_confident]
#             if active_sliding_window_mask is not None:
#                 active_sliding_window_mask = active_sliding_window_mask[not_confident]
#             active_pos_ids = active_pos_ids[not_confident]

#         return {"logits": final_logits}

## bert tiny

In [6]:
mg_baseline = MaseGraph.from_checkpoint("/vol/bitbucket/ug22/adls-data/models/bert-tiny-imdb-baseline")
baseline_model = mg_baseline.model.to(DEVICE)

# 1. Reconstruct the model (as you already did)
hf_model = AutoModelForSequenceClassification.from_pretrained("DeepWokLab/bert-tiny", num_labels=2)
hf_model.load_state_dict(baseline_model.state_dict(), strict=False)
hf_model = hf_model.to(DEVICE)
print(hf_model.bert)
THRESHOLD = 0.75
print(f"Baseline Hidden Size: {baseline_model.config.hidden_size}")
print(f"HF Model Hidden Size: {hf_model.config.hidden_size}")
print("\n" + "="*50)
print(f"EARLY EXIT (Threshold = {THRESHOLD})")
print("="*50)

# 2. FIX: Call the dedicated BERT Adapter! No more string paths.
early_exit_model = BertEarlyExit(
    original_model=hf_model, 
    threshold=THRESHOLD
).to(DEVICE)

# 3. Now run your evaluation
eval_accuracy(early_exit_model, eval_dataloader, device=DEVICE)
eval_speed(early_exit_model, eval_dataloader, device=DEVICE)

# 4. baseline model for reference
print("\n" + "="*50)
print("BASELINE BERT-TINY (No Early Exit)")
print("="*50)
eval_accuracy(hf_model, eval_dataloader, device=DEVICE)
eval_speed(hf_model, eval_dataloader, device=DEVICE)

WARNING  Node finfo not found in loaded metadata.
WARNING  Node getattr_2 not found in loaded metadata.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepWokLab/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Baseline Hidden Size: 128
HF Model Hidden Size: 128

EARLY EXIT (Threshold = 0.75)


Eval Accuracy:   0%|          | 0/391 [00:00<?, ?it/s]/vol/bitbucket/nr722/adls-project/.venv/lib/python3.11/site-packages/transformers/modeling_utils.py:1575: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
Eval Accuracy: 100%|██████████| 391/391 [00:05<00:00, 65.69it/s]


Accuracy: 81.20% (20299/25000)
6400 samples in 0.2945s
Throughput:      21731.6 samples/sec
Avg batch:       2.95 ms
Avg per-sample:  0.05 ms

Loading baseline model...
BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 128, padding_idx=0)
    (position_embeddings): Embedding(512, 128)
    (token_type_embeddings): Embedding(2, 128)
    (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-1): 2 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=128, out_features=128, bias=True)
            (key): Linear(in_features=128, out_features=128, bias=True)
            (value): Linear(in_features=128, out_features=128, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_featur

Eval Accuracy: 100%|██████████| 391/391 [00:06<00:00, 63.33it/s]


Accuracy: 82.34% (20586/25000)
6400 samples in 0.4910s
Throughput:      13035.9 samples/sec
Avg batch:       4.91 ms
Avg per-sample:  0.08 ms


[np.float64(4.909501090023696), 0.07671095453162025]

## Modernbert

In [87]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "./final_modernbert_imdb_model" 
THRESHOLD = 1  # Set to 1.0 to verify it matches original accuracy

# A. Load Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
hf_model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(DEVICE)

# B. Prepare Dataset specifically for ModernBERT
dataset = load_dataset("imdb")

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

# Tokenize and clean columns (Important: No token_type_ids for ModernBERT)
tokenized_test = dataset["test"].map(tokenize_function, batched=True)
eval_ds = tokenized_test.remove_columns(["text"]).rename_column("label", "labels")
if "token_type_ids" in eval_ds.column_names:
    eval_ds = eval_ds.remove_columns(["token_type_ids"])

eval_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# C. Create Dataloader
eval_dataloader = DataLoader(
    eval_ds,
    batch_size=64,
    shuffle=False,
    collate_fn=DataCollatorWithPadding(tokenizer),
)


In [88]:

# D. Initialize and Run
early_exit_model = ModernBertEarlyExit(hf_model, threshold=THRESHOLD).to(DEVICE)

print("\n" + "="*40)
print(f"MODERNBERT EVALUATION (Threshold: {THRESHOLD})")
print("="*40)

eval_accuracy(early_exit_model, eval_dataloader, DEVICE)
# eval_speed(early_exit_model, eval_dataloader, DEVICE)

# print("\n" + "="*40)
# print("BASELINE MODERNBERT (No Early Exit)")
# print("="*40)
# # eval_accuracy(hf_model, eval_dataloader, DEVICE)
# eval_speed(hf_model, eval_dataloader, DEVICE)



MODERNBERT EVALUATION (Threshold: 1)


Eval Accuracy: 100%|██████████| 391/391 [03:21<00:00,  1.94it/s]

Accuracy: 95.50% (23876/25000)


0.95504

## Bert Base

In [ ]:
# # ==========================================
# hf_bert = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2).to(DEVICE)
# # hf_bert.load_state_dict(...) # Load your fine-tuned weights here if you have them

# THRESHOLD = 0.75
# print("\n" + "="*50)
# print(f"EVALUATING BERT BASE (Threshold = {THRESHOLD})")
# print("="*50)

# early_exit_bert = BertEarlyExit(hf_bert, threshold=THRESHOLD).to(DEVICE)

# eval_accuracy(early_exit_bert, eval_dataloader, device=DEVICE)
# eval_speed(early_exit_bert, eval_dataloader, device=DEVICE)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



EVALUATING BERT BASE (Threshold = 0.75)


Eval Accuracy:   0%|          | 0/391 [00:00<?, ?it/s]/vol/bitbucket/nr722/adls-project/.venv/lib/python3.11/site-packages/transformers/modeling_utils.py:1575: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
Eval Accuracy:  32%|███▏      | 126/391 [00:37<01:18,  3.36it/s]


KeyboardInterrupt: 

## Roberta

In [26]:

# 1. Load the MaseGraph baseline
# Note: Ensure this path points to the roberta-imdb-baseline folder
ROBERTA_MASE_PATH = "/vol/bitbucket/ug22/adls-data/models/roberta-imdb-baseline"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_CKPT = "/vol/bitbucket/ug22/adls-data/models/roberta-imdb-baseline"
TOKENIZER_CKPT = "roberta-base"

# Data
dataset, tokenizer = get_tokenized_dataset(dataset="imdb", checkpoint=TOKENIZER_CKPT, return_tokenizer=True)
collator = DataCollatorWithPadding(tokenizer)

def make_loader(split, bs, shuffle=False):
    ds = dataset[split].remove_columns(["text"]).rename_column("label", "labels")
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, collate_fn=collator)

eval_loader = make_loader("test", 64)


INFO     Tokenizing dataset imdb with AutoTokenizer for roberta-base.


In [ ]:
from transformers import AutoModelForSequenceClassification
import torch

# Ensure DEVICE is correctly typed for eval_speed
DEVICE_OBJ = torch.device(DEVICE) if isinstance(DEVICE, str) else DEVICE

# Load model baseline from MaseGraph
threshold = 0.75
mg = MaseGraph.from_checkpoint(MODEL_CKPT)
mase_model = mg.model.to(DEVICE_OBJ)

# Reconstruct the model using standard HuggingFace structure
# This is required because MaseGraph models (GraphModules) cannot be dynamically 
# sliced for Early Exit without breaking their internal computational graph.
hf_model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=2)
hf_model.load_state_dict(mase_model.state_dict(), strict=False)
hf_model = hf_model.to(DEVICE_OBJ).eval()

model = RobertaEarlyExit(hf_model, threshold=threshold).to(DEVICE_OBJ)

# 5. Run Eval
eval_accuracy(model, eval_loader, device=DEVICE_OBJ)
eval_speed(model, eval_loader, device=DEVICE_OBJ)

# 6. Baseline for reference
print("\n" + "="*50)
print("BASELINE ROBERTA (No Early Exit)")
print("="*50)
eval_accuracy(hf_model, eval_loader, device=DEVICE_OBJ)
eval_speed(hf_model, eval_loader, device=DEVICE_OBJ)

WARNING  Node finfo not found in loaded metadata.
WARNING  Node getattr_2 not found in loaded metadata.
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


6400 samples in 25.4852s
Throughput:      251.1 samples/sec
Avg batch:       254.85 ms
Avg per-sample:  3.98 ms

BASELINE ROBERTA (No Early Exit)
6400 samples in 28.1349s
Throughput:      227.5 samples/sec
Avg batch:       281.35 ms
Avg per-sample:  4.40 ms


[np.float64(281.348560149745), 4.396071252339766]

In [ ]:
import matplotlib.pyplot as plt

thresholds = [0.5, 0.65, 0.7, 0.73, 0.75, 0.77, 0.8, 0.9]
accuracies = []
latencies = []

print("Evaluating different confidence thresholds...")
for t in thresholds:
    print(f"\n--- Threshold: {t} ---")
    ee_model = EarlyExitBert(hf_model, threshold=t).to(DEVICE)
    acc = eval_accuracy(ee_model, eval_dataloader, device=DEVICE)
    lat, _ = eval_speed(ee_model, eval_dataloader, device=DEVICE)
    accuracies.append(acc * 100)
    latencies.append(lat)

# Get baseline for comparison
print(f"\n--- Baseline Model ---")
base_acc = eval_accuracy(baseline_model, eval_dataloader, device=DEVICE) * 100
base_lat, _ = eval_speed(baseline_model, eval_dataloader, device=DEVICE)

# Simplified Plotting
plt.figure(figsize=(9, 6))

# Plot Early Exit points
plt.scatter(latencies, accuracies, color='b', s=100, label='Early Exit')
plt.plot(latencies, accuracies, 'b--', alpha=0.5)

for i, t in enumerate(thresholds):
    plt.annotate(f"T={t}", (latencies[i], accuracies[i]), 
                 textcoords="offset points", xytext=(0, 10), ha='center')

# Plot Baseline
plt.scatter([base_lat], [base_acc], color='r', marker='*', s=200, label='Baseline')
plt.annotate("Baseline", (base_lat, base_acc), 
             textcoords="offset points", xytext=(0, -15), ha='center', color='red')

plt.title('Performance vs Efficiency Trade-off: Accuracy vs Latency')
plt.xlabel('Average Inference Latency per batch (ms)')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)
plt.show()